# 🥈 Silver Layer: Data Cleansing & Historical Tracking

**Objective:** Cleanse the raw Bronze data, enforce data quality, and establish a validated enterprise source of truth.

### Architecture Details
* **Data Quality (Deduplication):** The incoming Bronze streams are strictly deduplicated based on the defined primary keys for each dataset.
* **SCD Type 2 Implementation:** We utilize Delta Lake's `MERGE INTO` (Upsert) capabilities to track historical state changes. Instead of overwriting old records, we maintain a history using `is_active`, `valid_from`, and `valid_to` columns.
* **Idempotent Execution:** The logic is designed to safely handle both initial bulk loads and ongoing incremental merges without duplicating data.

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit
from delta.tables import DeltaTable

# Your Unity Catalog external location
base_path = "abfss://iaspcontainer@iaspstorage.dfs.core.windows.net"

# Defining Primary Keys for each dataset to handle SCD2 Merges
primary_keys = {
    "products": "product_id",
    "inventory": "inventory_id",
    "suppliers": "supplier_id",
    "warehouses": "warehouse_id",
    "transactions": "transaction_id",
    "shipments": "shipment_id"
}

print("Initializing Silver Layer Transformations (SCD Type 2)...")

for dataset, pk in primary_keys.items():
    bronze_path = f"{base_path}/bronze/{dataset}/"
    silver_path = f"{base_path}/silver/{dataset}/"
    
    print(f"Cleaning and processing SCD2 for: {dataset} (Key: {pk})")
    
    try:
        # 1. Read Bronze Data and enforce deduplication based on Primary Key
        bronze_df = spark.read.format("delta").load(bronze_path).dropDuplicates([pk])
        
        # 2. Add SCD Type 2 Tracking Columns
        clean_df = (bronze_df
                    .withColumn("is_active", lit(True))
                    .withColumn("valid_from", current_timestamp())
                    .withColumn("valid_to", lit(None).cast("timestamp")))
        
        # 3. Perform Delta Merge (Upsert) to maintain historical changes
        if DeltaTable.isDeltaTable(spark, silver_path):
            silver_table = DeltaTable.forPath(spark, silver_path)
            
            # Merge incoming data with existing Silver data
            silver_table.alias("target").merge(
                clean_df.alias("source"),
                f"target.{pk} = source.{pk}"
            ).whenMatchedUpdateAll(
            ).whenNotMatchedInsertAll().execute()
            
        else:
            # Initial Load: If the Silver table doesn't exist yet, create it
            (clean_df.write
             .format("delta")
             .mode("overwrite")
             .option("path", silver_path)
             .saveAsTable(f"silver_{dataset}"))
            
    except Exception as e:
        print(f"Error processing {dataset}: {e}")

print("Silver Layer Cleansing & SCD2 Implementation Complete!")

Initializing Silver Layer Transformations (SCD Type 2)...
Cleaning and processing SCD2 for: products (Key: product_id)
Cleaning and processing SCD2 for: inventory (Key: inventory_id)
Cleaning and processing SCD2 for: suppliers (Key: supplier_id)
Cleaning and processing SCD2 for: warehouses (Key: warehouse_id)
Cleaning and processing SCD2 for: transactions (Key: transaction_id)
Cleaning and processing SCD2 for: shipments (Key: shipment_id)
Silver Layer Cleansing & SCD2 Implementation Complete!


In [0]:
%sql
SELECT * FROM silver_products LIMIT 5;

product_id,product_name,category,brand,cost_price,selling_price,currency,supplier_id,product_status,launch_date,expiry_date,weight,dimensions,reorder_level,last_updated,_rescued_data,ingestion_timestamp,source_file_name,silver_update_timestamp
P1157,Stay,Clothing>Men>Topwear,Dell,48503.95,21513.25,INR,S208,inactive,2023-07-24,null,4.94,34x36x3,189,2025-12-21,null,2026-07-11T17:23:24.139Z,abfss://iaspcontainer@iaspstorage.dfs.core.windows.net/raw/products/products.csv,2026-07-11T17:25:49.944Z
P1215,Audience,Electronics>Mobile>Accessories,Boat,43487.03,2851.14,INR,S205,active,2024-04-16,null,3.57,6x16x1,147,2025-11-20,null,2026-07-11T17:23:24.139Z,abfss://iaspcontainer@iaspstorage.dfs.core.windows.net/raw/products/products.csv,2026-07-11T17:25:49.944Z
P1010,Decision,Electronics>Mobile>Accessories,null,28434.89,63480.33,INR,S207,discontinued,2024-11-24,null,2.99,44x13x5,77,2025-04-21,null,2026-07-11T17:23:24.139Z,abfss://iaspcontainer@iaspstorage.dfs.core.windows.net/raw/products/products.csv,2026-07-11T17:25:49.944Z
P1279,Open,Electronics>Mobile>Accessories,Boat,35897.19,51618.96,INR,S210,active,2024-10-02,2027-03-26,4.08,32x42x1,87,2025-12-20,null,2026-07-11T17:23:24.139Z,abfss://iaspcontainer@iaspstorage.dfs.core.windows.net/raw/products/products.csv,2026-07-11T17:25:49.944Z
P1362,That,Clothing>Men>Topwear,Boat,24979.46,18767.79,INR,S210,discontinued,2023-11-25,null,4.47,27x27x9,131,2026-02-06,null,2026-07-11T17:23:24.139Z,abfss://iaspcontainer@iaspstorage.dfs.core.windows.net/raw/products/products.csv,2026-07-11T17:25:49.944Z
